In [116]:
import pandas as pd

from DWG_to_DuckDB_ETL import *
import os
import shutil

dwg_folder = r"C:\Users\Usuario\Desktop\projetos\paiva\teste"
temp_folder = r"C:\Users\Usuario\Desktop\projetos\paiva\temp"
output_folder = dwg_folder

os.makedirs(temp_folder, exist_ok=True)
dwg_file = os.path.join(dwg_folder, "PV Paiva - Eletromecânico NextPower - Tags.dwg")

shutil.copy(dwg_file, temp_folder)

convert_dwg_to_dxf(temp_folder, output_folder)

dxf_file = os.path.join(output_folder, "PV Paiva - Eletromecânico NextPower - Tags.dxf")

temp_dwg_file = os.path.join(temp_folder, "PV Paiva - Eletromecânico NextPower - Tags.dwg")

conn = load_to_duckdb(dxf_file, "PV Paiva")
print("Data Loaded")
print("DuckDB Loaded")


os.remove(dxf_file)
os.remove(temp_dwg_file)

Project PV Paiva already exists
Removed existing data for project: PV Paiva
Successfully loaded project
Loaded 12852 entities for project: PV Paiva
Don't forget to close connection.
Data Loaded
DuckDB Loaded


In [117]:
data = conn.sql("SELECT type, layer, name, color, rotation, X, Y, UFV, PST, INV, STR, GRP FROM cad_entities WHERE project_name = 'PV Paiva'")
print(data)


┌─────────┬────────────────────┬──────────────────────────────────┬─────────┬──────────┬────────────┬────────────┬─────────┬─────────┬─────────┬─────────┬─────────┐
│  type   │       layer        │               name               │  color  │ rotation │     X      │     Y      │   UFV   │   PST   │   INV   │   STR   │   GRP   │
│ varchar │      varchar       │             varchar              │ varchar │  float   │   float    │   float    │ varchar │ varchar │ varchar │ varchar │ varchar │
├─────────┼────────────────────┼──────────────────────────────────┼─────────┼──────────┼────────────┼────────────┼─────────┼─────────┼─────────┼─────────┼─────────┤
│ INSERT  │ 0                  │ EMF397_PST                       │ 256     │     90.0 │  34592.234 │   127259.4 │         │ 22      │ NULL    │ NULL    │ NULL    │
│ INSERT  │ 0                  │ EMF397_PST                       │ 256     │     90.0 │ 34995.4375 │   127259.4 │         │ 23      │ NULL    │ NULL    │ NULL    │
│ INSERT  

In [118]:
result = conn.sql("""
    SELECT
        layer,
        COUNT(*) as count
    FROM data
    GROUP BY layer
    ORDER BY count
""")
print(result)

┌────────────────────┬───────┐
│       layer        │ count │
│      varchar       │ int64 │
├────────────────────┼───────┤
│ EMF_em_pst_26      │    14 │
│ EMF_em_pst_25      │    15 │
│ EMF_em_pst_21      │    16 │
│ EMF_em_pst_27      │    16 │
│ EMF_em_pst_19      │    16 │
│ EMF_em_pst_02      │    16 │
│ EMF_em_pst_22      │    16 │
│ EMF_em_pst_06      │    16 │
│ EMF_em_pst_24      │    16 │
│ EMF_em_pst_18      │    16 │
│       ·            │     · │
│       ·            │     · │
│       ·            │     · │
│ EMF_em_pst_01      │    16 │
│ EMF_em_pst_17      │    16 │
│ 0                  │    28 │
│ EMF_em_pst_13      │   120 │
│ EMF_em_pst_04      │   150 │
│ EMF_em_pst_07      │   442 │
│ EMF_em_pst_09      │   442 │
│ EMF_em_tracker_32p │   948 │
│ EMF_em_tracker_64p │  2672 │
│ EMF_em_tracker_96p │  7653 │
└────────────────────┴───────┘
  33 rows          2 columns
  (20 shown)                 



In [119]:
inverters = conn.sql("SELECT * FROM data WHERE name LIKE '%inversor%'")
inverters_df = inverters.fetchdf()
inverters_df['UFV'] = 0
inverters_df['PST'] = inverters_df['layer'].astype(str).str[-2:]
inverters_df['X_ordering'] = round(inverters_df['X'], 1)
inverters_df['Y_ordering'] = round(inverters_df['Y'], 1)
inverters_df['color'] = inverters_df['color'].astype(int)
print(inverters_df)

       type          layer             name  color  rotation             X  \
0    INSERT  EMF_em_pst_29  EMF397_inversor    256     180.0  34286.687500   
1    INSERT  EMF_em_pst_29  EMF397_inversor    256     180.0  34368.585938   
2    INSERT  EMF_em_pst_29  EMF397_inversor    256     180.0  34463.085938   
3    INSERT  EMF_em_pst_29  EMF397_inversor    256     180.0  34494.585938   
4    INSERT  EMF_em_pst_29  EMF397_inversor    256     180.0  34538.687500   
..      ...            ...              ...    ...       ...           ...   
457  INSERT  EMF_em_pst_15  EMF397_inversor    256     180.0  34368.585938   
458  INSERT  EMF_em_pst_15  EMF397_inversor    256     180.0  34425.285156   
459  INSERT  EMF_em_pst_09  EMF397_inversor      9       0.0  34291.253906   
460  INSERT  EMF_em_pst_04  EMF397_inversor      4       0.0  34335.351562   
461  INSERT  EMF_em_pst_09  EMF397_inversor      4       0.0  34272.355469   

                 Y  UFV PST INV STR   GRP    X_ordering     Y_o

In [120]:
inverters_df['GRP'] = inverters_df["UFV"].astype(str) + "." + inverters_df["PST"].astype(str)
inverters_df.sort_values(by=['GRP', 'Y_ordering', 'X_ordering'], ascending=[True, False, True], inplace=True, ignore_index=True)
inverters_df["INV"] = (inverters_df.groupby("GRP").cumcount()+1).astype(str).str.zfill(2)
inverters_df['grp.color'] = inverters_df['GRP'].astype(str) + "." + inverters_df['color'].astype(str)

inverters_df['Script'] = inverters_df.apply(
    lambda row: (
        f'(command-s "._insert" "{row['name']}" '
        f'"{row["X"]},{row["Y"]}" 1 {row['rotation']} {str(row["UFV"]).zfill(2)} {str(row["PST"]).zfill(2)} {str(row["INV"]).zfill(2)} {str(row["STR"]).zfill(2)})'
    ),
    axis=1
)

print(inverters_df)

       type          layer             name  color  rotation             X  \
0    INSERT  EMF_em_pst_01  EMF397_inversor    256     180.0  36054.707031   
1    INSERT  EMF_em_pst_01  EMF397_inversor    256     180.0  35947.609375   
2    INSERT  EMF_em_pst_01  EMF397_inversor    256     180.0  35827.906250   
3    INSERT  EMF_em_pst_01  EMF397_inversor    256     180.0  35752.304688   
4    INSERT  EMF_em_pst_01  EMF397_inversor    256     180.0  35443.605469   
..      ...            ...              ...    ...       ...           ...   
457  INSERT  EMF_em_pst_29  EMF397_inversor    256     180.0  34589.085938   
458  INSERT  EMF_em_pst_29  EMF397_inversor    256     180.0  34645.785156   
459  INSERT  EMF_em_pst_29  EMF397_inversor    256       0.0  34544.988281   
460  INSERT  EMF_em_pst_29  EMF397_inversor    256     180.0  34286.687500   
461  INSERT  EMF_em_pst_29  EMF397_inversor    256     180.0  34368.585938   

                 Y  UFV PST INV STR   GRP    X_ordering     Y_o

In [121]:
strings = conn.sql("SELECT * FROM data WHERE name LIKE '%Tracker%'")
strings_df = strings.fetchdf()
strings_df['UFV'] = 0
strings_df['PST'] = strings_df['layer'].astype(str).str[-2:]
print(strings_df)

         type               layer                              name color  \
0      INSERT  EMF_em_tracker_64p        String_Tracker_1x64p_North     3   
1      INSERT  EMF_em_tracker_64p        String_Tracker_1x64p_North     3   
2      INSERT  EMF_em_tracker_64p        String_Tracker_1x64p_North     3   
3      INSERT  EMF_em_tracker_64p        String_Tracker_1x64p_North     3   
4      INSERT  EMF_em_tracker_64p        String_Tracker_1x64p_North     3   
...       ...                 ...                               ...   ...   
12357  INSERT       EMF_em_pst_07  String_Tracker_1x96p_Ponta_North   256   
12358  INSERT  EMF_em_tracker_64p        String_Tracker_1x64p_North     3   
12359  INSERT  EMF_em_tracker_64p        String_Tracker_1x64p_North     3   
12360  INSERT  EMF_em_tracker_64p        String_Tracker_1x64p_North     3   
12361  INSERT  EMF_em_tracker_64p        String_Tracker_1x64p_North     3   

       rotation             X              Y  UFV PST INV STR   GRP  
0    

In [122]:
strings9 = strings_df[strings_df['layer'] == 'EMF_em_pst_09']
strings9['grp.color'] = strings9['UFV'].astype(str) + "." + strings9['PST'].astype(str) + '.' + strings9['color'].astype(str)
strings9.drop(columns=['INV'], inplace=True)
print(strings9)

         type          layer                        name color  rotation  \
67     INSERT  EMF_em_pst_09  String_Tracker_1x64p_North    16       0.0   
68     INSERT  EMF_em_pst_09  String_Tracker_1x64p_North     5       0.0   
69     INSERT  EMF_em_pst_09  String_Tracker_1x64p_North    16       0.0   
70     INSERT  EMF_em_pst_09  String_Tracker_1x64p_North    16       0.0   
71     INSERT  EMF_em_pst_09  String_Tracker_1x64p_North    16       0.0   
...       ...            ...                         ...   ...       ...   
12319  INSERT  EMF_em_pst_09  String_Tracker_1x64p_North     9     180.0   
12320  INSERT  EMF_em_pst_09  String_Tracker_1x32p_North     7       0.0   
12321  INSERT  EMF_em_pst_09  String_Tracker_1x32p_North     7       0.0   
12322  INSERT  EMF_em_pst_09  String_Tracker_1x32p_North     7       0.0   
12346  INSERT  EMF_em_pst_09  String_Tracker_1x32p_North    16       0.0   

                  X              Y  UFV PST STR   GRP grp.color  
67     34070.753906  

In [123]:
inverters_subset = inverters_df[['grp.color', 'INV']]
emf9 = strings9.merge(inverters_subset, on='grp.color', how='left')
print(emf9.columns)
print(emf9)
emf9.to_excel(r"C:\Users\Usuario\Desktop\emf9.xlsx")

Index(['type', 'layer', 'name', 'color', 'rotation', 'X', 'Y', 'UFV', 'PST',
       'STR', 'GRP', 'grp.color', 'INV'],
      dtype='str')
       type          layer                        name color  rotation  \
0    INSERT  EMF_em_pst_09  String_Tracker_1x64p_North    16       0.0   
1    INSERT  EMF_em_pst_09  String_Tracker_1x64p_North     5       0.0   
2    INSERT  EMF_em_pst_09  String_Tracker_1x64p_North    16       0.0   
3    INSERT  EMF_em_pst_09  String_Tracker_1x64p_North    16       0.0   
4    INSERT  EMF_em_pst_09  String_Tracker_1x64p_North    16       0.0   
..      ...            ...                         ...   ...       ...   
421  INSERT  EMF_em_pst_09  String_Tracker_1x64p_North     9     180.0   
422  INSERT  EMF_em_pst_09  String_Tracker_1x32p_North     7       0.0   
423  INSERT  EMF_em_pst_09  String_Tracker_1x32p_North     7       0.0   
424  INSERT  EMF_em_pst_09  String_Tracker_1x32p_North     7       0.0   
425  INSERT  EMF_em_pst_09  String_Tracker_1x32p

In [124]:
emf9['GRP'] = emf9["UFV"].astype(str) + "." + emf9["PST"].astype(str) + "." + emf9["INV"].astype(str)
emf9['X_ordering'] = round(emf9['X'], 1)
emf9['Y_ordering'] = round(emf9['Y'], 1)
emf9.sort_values(by=['GRP', 'X_ordering', 'Y_ordering'], ascending=[True, True, False], inplace=True, ignore_index=True)
emf9["STR"] = (emf9.groupby("GRP").cumcount()+1).astype(str).str.zfill(2)


emf9['Script'] = emf9.apply(
    lambda row: (
        f'(command-s "._insert" "{row['name']}" '
        f'"{row["X"]},{row["Y"]}" 1 1 {row["rotation"]} {str(row["UFV"]).zfill(2)} {str(row["PST"]).zfill(2)} {str(row["INV"]).zfill(2)} {str(row["STR"]).zfill(2)})'
    ),
    axis=1
)
print(emf9)

       type          layer                              name color  rotation  \
0    INSERT  EMF_em_pst_09   String_Tracker_1x96p_Meio_North     1       0.0   
1    INSERT  EMF_em_pst_09  String_Tracker_1x96p_Ponta_North     1     180.0   
2    INSERT  EMF_em_pst_09        String_Tracker_1x64p_North     1       0.0   
3    INSERT  EMF_em_pst_09        String_Tracker_1x64p_North     1     180.0   
4    INSERT  EMF_em_pst_09        String_Tracker_1x64p_North     1       0.0   
..      ...            ...                               ...   ...       ...   
421  INSERT  EMF_em_pst_09   String_Tracker_1x96p_Meio_North    16       0.0   
422  INSERT  EMF_em_pst_09  String_Tracker_1x96p_Ponta_North    16     180.0   
423  INSERT  EMF_em_pst_09  String_Tracker_1x96p_Ponta_North    16       0.0   
424  INSERT  EMF_em_pst_09   String_Tracker_1x96p_Meio_North    16       0.0   
425  INSERT  EMF_em_pst_09  String_Tracker_1x96p_Ponta_North    16     180.0   

                X              Y  UFV P

In [125]:
with open(r"C:\Users\Usuario\Desktop\Pessoal\Data Engineering\Python\Ecotechne\Tagueamento Inversor e Estruturas\Reinserting Trackers - Paiva.scr", 'w') as f:
    for i in emf9['layer'].unique():
        mask = emf9['layer'] == i

        if mask.any():
            f.write(f'(command-s "._layer" "set" "{i}" "")\n')
            f.write('\n'.join(emf9[mask]['Script'].astype(str)) + '\n')

with open(r"C:\Users\Usuario\Desktop\Pessoal\Data Engineering\Python\Ecotechne\Tagueamento Inversor e Estruturas\Reinserting Inverters - Paiva.scr", 'w') as f:
    for i in inverters_df['layer'].unique():
        mask = inverters_df['layer'] == i

        if mask.any():
            f.write(f'(command-s "._layer" "set" "{i}" "")\n')
            f.write('\n'.join(inverters_df[mask]['Script'].astype(str)) + '\n')


conn.close()